# ⚡ Notebook 4: Temporal Basics

Introduction to durable execution with Temporal.

## Learning Objectives

By the end of this notebook, you'll understand:
- What Temporal is and why it exists
- Workflows vs Activities
- How to write a basic workflow
- Running workflows with Python SDK

In [ ]:
import asyncio
from datetime import timedelta
from temporalio import activity, workflow
from temporalio.client import Client
from temporalio.common import RetryPolicy
from temporalio.worker import Worker
import uuid

print("✅ Temporal SDK imported!")

## ⚡ What is Temporal?

In [ ]:
print("⚡ Temporal = Durable Execution Engine")
print("=" * 60)
print("""
Temporal lets you write workflows as NORMAL CODE:

┌─────────────────────────────────────────────────────────────┐
│  async def order_workflow(order):                           │
│      payment = await charge_payment(order)      # Step 1    │
│      inventory = await reserve_inventory(order) # Step 2    │
│      shipping = await create_shipping(order)    # Step 3    │
│      return {"tracking": shipping.tracking}                 │
└─────────────────────────────────────────────────────────────┘

Looks like normal Python, BUT:
• If server crashes, workflow continues from last step
• If step fails, automatic retries with backoff
• Can wait for hours/days without blocking resources
• Full history visible in Temporal UI

HOW IT WORKS:
─────────────────────────────────────────────────────────────
1. Workflow code runs on workers (your servers)
2. Every activity result is saved to Temporal server
3. If worker crashes, another replays workflow
4. Replay skips completed activities (uses saved results)

           Your Code                    Temporal Server
         ─────────────                ─────────────────
         [Workflow]  ◄─── results ───  [History DB]
         [Activities]                  [Event Log]
""")

## 📝 Workflows vs Activities

In [ ]:
print("📝 Key Concepts")
print("=" * 60)
print("""
WORKFLOW = The orchestration logic
─────────────────────────────────────────────────────────────
• Defines the order of steps
• Must be DETERMINISTIC (no random, no time.now())
• Can run for days/months/years
• Replayed on crash to rebuild state

Think of it as: "First do X, then do Y, if Y fails do Z"

ACTIVITY = Individual steps with side effects
─────────────────────────────────────────────────────────────
• Actually calls external services
• CAN have side effects (DB writes, API calls)
• Should be IDEMPOTENT (safe to retry)
• Results are saved by Temporal

Think of it as: "Call Stripe API", "Update database"

WHY THE SPLIT?
─────────────────────────────────────────────────────────────
Workflow:  charge → reserve → ship   (deterministic logic)
                │        │       │
Activities:     ▼        ▼       ▼
           [Stripe]  [DB]   [FedEx]   (actual I/O)

On replay:
• Workflow logic runs again (deterministic = same decisions)
• Activity results are loaded from history (not re-executed)
""")

## 🔧 Defining Activities

In [ ]:
from dataclasses import dataclass
import random
import time

@dataclass
class OrderInput:
    order_id: str
    amount: float
    item_sku: str

@dataclass
class PaymentResult:
    transaction_id: str
    amount: float

@dataclass
class InventoryResult:
    reservation_id: str
    sku: str

@dataclass
class ShippingResult:
    tracking_number: str

@activity.defn
async def charge_payment(order: OrderInput) -> PaymentResult:
    print(f"   💳 Charging ${order.amount} for order {order.order_id[:8]}...")
    await asyncio.sleep(0.5)
    
    if random.random() < 0.2:
        raise Exception("Payment gateway timeout")
    
    return PaymentResult(
        transaction_id=f"txn_{uuid.uuid4().hex[:8]}",
        amount=order.amount
    )

@activity.defn
async def reserve_inventory(order: OrderInput) -> InventoryResult:
    print(f"   📦 Reserving {order.item_sku} for order {order.order_id[:8]}...")
    await asyncio.sleep(0.3)
    
    return InventoryResult(
        reservation_id=f"res_{uuid.uuid4().hex[:8]}",
        sku=order.item_sku
    )

@activity.defn
async def create_shipping(order: OrderInput) -> ShippingResult:
    print(f"   🚚 Creating shipping label for order {order.order_id[:8]}...")
    await asyncio.sleep(0.4)
    
    return ShippingResult(
        tracking_number=f"1Z{uuid.uuid4().hex[:12].upper()}"
    )

@activity.defn
async def send_confirmation(order_id: str, tracking: str) -> bool:
    print(f"   ✉️ Sending confirmation email for order {order_id[:8]}...")
    await asyncio.sleep(0.2)
    return True

print("✅ Activities defined!")

## 🎯 Defining the Workflow

In [ ]:
@dataclass
class OrderResult:
    order_id: str
    transaction_id: str
    reservation_id: str
    tracking_number: str
    status: str

@workflow.defn
class OrderWorkflow:
    @workflow.run
    async def run(self, order: OrderInput) -> OrderResult:
        print(f"\n🚀 Starting workflow for order {order.order_id[:8]}")
        
        payment = await workflow.execute_activity(
            charge_payment,
            order,
            start_to_close_timeout=timedelta(seconds=30),
            retry_policy=RetryPolicy(
                maximum_attempts=3,
                initial_interval=timedelta(seconds=1),
                backoff_coefficient=2.0
            )
        )
        print(f"   ✅ Payment: {payment.transaction_id}")
        
        inventory = await workflow.execute_activity(
            reserve_inventory,
            order,
            start_to_close_timeout=timedelta(seconds=30),
        )
        print(f"   ✅ Inventory: {inventory.reservation_id}")
        
        shipping = await workflow.execute_activity(
            create_shipping,
            order,
            start_to_close_timeout=timedelta(seconds=30),
        )
        print(f"   ✅ Shipping: {shipping.tracking_number}")
        
        await workflow.execute_activity(
            send_confirmation,
            args=[order.order_id, shipping.tracking_number],
            start_to_close_timeout=timedelta(seconds=30),
        )
        print(f"   ✅ Confirmation sent!")
        
        return OrderResult(
            order_id=order.order_id,
            transaction_id=payment.transaction_id,
            reservation_id=inventory.reservation_id,
            tracking_number=shipping.tracking_number,
            status="completed"
        )

print("✅ OrderWorkflow defined!")

## 🏃 Running the Workflow

In [ ]:
async def run_order_workflow():
    client = await Client.connect("localhost:7233")
    print("✅ Connected to Temporal!")
    
    async with Worker(
        client,
        task_queue="order-queue",
        workflows=[OrderWorkflow],
        activities=[charge_payment, reserve_inventory, create_shipping, send_confirmation]
    ):
        print("\n👷 Worker started!")
        
        order = OrderInput(
            order_id=str(uuid.uuid4()),
            amount=99.99,
            item_sku="WIDGET-X"
        )
        
        result = await client.execute_workflow(
            OrderWorkflow.run,
            order,
            id=f"order-{order.order_id[:8]}",
            task_queue="order-queue"
        )
        
        print(f"\n📊 Workflow Result:")
        print(f"   Order ID: {result.order_id[:8]}...")
        print(f"   Transaction: {result.transaction_id}")
        print(f"   Reservation: {result.reservation_id}")
        print(f"   Tracking: {result.tracking_number}")
        print(f"   Status: {result.status}")
        
        return result

print("⏳ Running workflow (make sure docker-compose is up!)")
print("=" * 60)

try:
    result = await run_order_workflow()
    print("\n✅ Workflow completed successfully!")
except Exception as e:
    print(f"\n❌ Error: {e}")
    print("   Make sure: docker compose up -d")

## 🖥️ Temporal Web UI

Open http://localhost:8080 to see:
- Running workflows
- Workflow history (every step!)
- Activity inputs and outputs
- Retry attempts

In [ ]:
print("🖥️ Temporal Web UI")
print("=" * 60)
print("""
Open: http://localhost:8080

You'll see:
┌─────────────────────────────────────────────────────────────┐
│  Namespace: default                                         │
├─────────────────────────────────────────────────────────────┤
│  Workflow ID          Status      Started      Duration     │
│  ─────────────────────────────────────────────────────────  │
│  order-abc12345       Completed   2 min ago    1.5s        │
│  order-def67890       Running     30 sec ago   ...         │
└─────────────────────────────────────────────────────────────┘

Click on a workflow to see:
• Event History: Every activity execution
• Input/Output: What was passed and returned
• Retries: How many times each activity was tried
• Stack Trace: Where the workflow is currently

This is MUCH better than digging through logs!
""")

## 🧪 Quick Quiz

1. **What's the difference between a workflow and an activity?**

2. **Why must workflows be deterministic?**

3. **What happens when an activity fails?**

In [ ]:
print("📝 Quiz Answers")
print("=" * 50)
print()
print("1. Workflow vs Activity:")
print("   - Workflow: orchestration logic (deterministic)")
print("   - Activity: actual work with side effects")
print("   - Workflow decides, Activity executes")
print()
print("2. Why deterministic workflows:")
print("   - Must replay to rebuild state")
print("   - Same inputs = same decisions")
print("   - Otherwise replay would diverge")
print()
print("3. When activity fails:")
print("   - Temporal retries based on policy")
print("   - Exponential backoff")
print("   - After max retries, workflow can handle")

## 📚 Summary

### What Temporal Provides

1. **Write workflows as code** - Not YAML or JSON
2. **Automatic persistence** - State saved to history
3. **Automatic retries** - Built-in retry policies
4. **Crash recovery** - Replay from history
5. **Full visibility** - Web UI shows everything

### Key Concepts

| Concept | Purpose |
|---------|--------|
| Workflow | Orchestration logic (deterministic) |
| Activity | Individual steps (side effects) |
| Worker | Executes workflows and activities |
| History | Record of all events for replay |

### Next Up

In **Notebook 5**, we'll dive into crash recovery:
- Heartbeats for long activities
- What happens when workers die
- How Temporal detects and recovers